# Fase 5 · Contrastes de hipótesis y sensibilidad
**Responsabilidad:** pruebas estadísticas H1 a H5, post-hoc y análisis de sensibilidad (un anuncio por anfitrión).

| Hipótesis | Prueba |
|---|---|
| H1 precio por localidad | Kruskal-Wallis + Dunn con Holm |
| H2 precio por tipo de alojamiento | Kruskal-Wallis + Dunn con Holm; Mann-Whitney completa vs. privada |
| H3 capacidad ↔ precio | Spearman |
| H4 calificación ↔ precio | Spearman |
| H5 anfitriones con varios anuncios vs. uno | Mann-Whitney por indicador, Holm en 4 indicadores (umbrales 2 y 5) |

| | |
|---|---|
| **Entrada** | salida de la fase 3 |
| **Salida** | `hipotesis.csv`, `descriptivos_H1/H2.csv`, `dunn_holm_H1/H2.csv`, `comparaciones_anfitriones.csv`, `sensibilidad_un_anuncio_por_host.csv`, `resumen_sensibilidad.csv` (en `resultados/`) |

## 1. Carga

In [1]:
import numpy as np
import pandas as pd

from bi_airbnb import (ALPHA, DIR_RESULTADOS, N_ITER_SENSIBILIDAD, SEED, cargar_base,
                       descriptivos, dunn, guardar_csv, holm, kruskal, mann_whitney, spearman)

todo, d, hosts = cargar_base()
filas = []   # una fila por contraste, en el orden de presentación de hipotesis.csv

## 2. H1 y H2: Kruskal-Wallis, descriptivos y Dunn-Holm
Solo grupos con n ≥ 30.

In [2]:
for hip, grupo in [("H1", "localidad"), ("H2", "tipo_alojamiento")]:
    sub, h, p, eta, k = kruskal(d, grupo)
    filas.append([hip, "Kruskal-Wallis", len(sub), h, p, p, "Sin ajuste global", "eta_H2", eta])
    guardar_csv(descriptivos(sub, grupo), DIR_RESULTADOS, f"descriptivos_{hip}.csv")
    guardar_csv(dunn(sub, grupo), DIR_RESULTADOS, f"dunn_holm_{hip}.csv")
    print(f"{hip}: {k} grupos, n = {len(sub):,}, H = {h:.1f}, p = {p:.3g}, eta² = {eta:.3f}")

H1: 16 grupos, n = 17,945, H = 2304.1, p = 0, eta² = 0.128
H2: 3 grupos, n = 17,971, H = 4159.8, p = 0, eta² = 0.231


### H2 (comparación prevista): vivienda completa vs. habitación privada

In [3]:
completa = d.loc[d.tipo_alojamiento.eq("Vivienda completa"), "precio_cop"]
privada = d.loc[d.tipo_alojamiento.eq("Habitación privada"), "precio_cop"]
u, p, delta = mann_whitney(completa, privada)
filas.append(["H2 completa vs privada", "Mann-Whitney", len(completa) + len(privada), u, p, p,
              "Comparación prevista", "delta_Cliff", delta])
print(f"U = {u:,.0f}, p = {p:.3g}, delta de Cliff = {delta:.3f}")

U = 50,270,100, p = 0, delta de Cliff = 0.623


## 3. H3 y H4: correlación de Spearman con el precio

In [4]:
for hip, variable in [("H3", "capacidad"), ("H4", "calificacion")]:
    n, rho, p = spearman(d, variable)
    filas.append([hip, "Spearman", n, rho, p, p, "Sin ajuste global", "rho", rho])
    print(f"{hip} ({variable}): n = {n:,}, rho = {rho:.3f}, p = {p:.3g}")

H3 (capacidad): n = 17,998, rho = 0.480, p = 0
H4 (calificacion): n = 14,549, rho = 0.046, p = 2.38e-08


## 4. H5: anfitriones con varios anuncios vs. un anuncio
Una observación por anfitrión. Holm sobre los 4 indicadores de cada umbral; la decisión de H5 usa el umbral 2.

In [5]:
INDICADORES_H5 = ["precio_mediano", "ocupacion_media_dias", "resenas_12m_media", "calificacion_mediana"]


def comparar_anfitriones(hosts, umbral):
    varios = hosts.loc[hosts.n_anuncios.ge(umbral)]
    uno = hosts.loc[hosts.n_anuncios.eq(1)]
    resultados = []
    for metrica in INDICADORES_H5:
        a, b = varios[metrica].dropna(), uno[metrica].dropna()
        u, p, delta = mann_whitney(a, b)
        resultados.append([umbral, metrica, len(a), len(b), a.median(), b.median(), u, p, delta])
    res = pd.DataFrame(resultados, columns=["umbral_varios", "metrica", "n_varios", "n_uno",
                                            "mediana_varios", "mediana_uno", "u", "p_bruto", "delta_Cliff"])
    res.insert(8, "p_holm", holm(res.p_bruto))
    return res


host_results = pd.concat([comparar_anfitriones(hosts, c) for c in (2, 5)], ignore_index=True)
guardar_csv(host_results, DIR_RESULTADOS, "comparaciones_anfitriones.csv")

for r in host_results[host_results.umbral_varios.eq(2)].itertuples():
    filas.append(["H5 " + r.metrica, "Mann-Whitney", r.n_varios + r.n_uno, r.u, r.p_bruto, r.p_holm,
                  "Holm en 4 indicadores", "delta_Cliff", r.delta_Cliff])
host_results

,umbral_varios,metrica,n_varios,n_uno,mediana_varios,mediana_uno,u,p_bruto,p_holm,delta_Cliff
0,2,precio_mediano,2378,5417,155584.100000,147799.00,6921843.5,1.454261e-07,1.454261e-07,0.074685
1,2,ocupacion_media_dias,2378,5417,36.428571,14.00,7630576.0,2.484011e-40,7.452033e-40,0.184722
2,2,resenas_12m_media,2378,5417,4.666667,2.00,7811660.0,5.672178e-53,2.268871e-52,0.212838
3,2,calificacion_mediana,2198,3973,4.830000,4.89,3495687.5,3.732473e-39,7.464945e-39,-0.199399
4,5,precio_mediano,751,5417,165117.000000,147799.00,2308755.5,1.899512e-09,1.899512e-09,0.135035
5,5,ocupacion_media_dias,751,5417,41.052632,14.00,2523594.5,2.915309e-28,5.830618e-28,0.240654
6,5,resenas_12m_media,751,5417,5.750000,2.00,2642111.5,1.089176e-42,4.356706e-42,0.298920
7,5,calificacion_mediana,734,3973,4.780000,4.89,1011803.5,1.681181e-40,5.043542e-40,-0.306077


## 5. Tabla de hipótesis
Un `p` numérico de 0 es subdesbordamiento, nunca una probabilidad exactamente nula; por eso se presenta como `< 0,001`.

In [6]:
tests = pd.DataFrame(filas, columns=["hipotesis", "prueba", "n", "estadistico", "p_bruto",
                                     "p_decision", "ajuste", "medida_efecto", "efecto"])
tests["decision"] = np.where(tests.p_decision < ALPHA, "Rechazar H0", "No rechazar H0")
tests["p_presentacion"] = tests.p_decision.map(lambda p: "< 0,001" if p < .001 else f"{p:.4f}")
guardar_csv(tests, DIR_RESULTADOS, "hipotesis.csv")
tests[["hipotesis", "prueba", "p_presentacion", "decision", "efecto"]]

,hipotesis,prueba,p_presentacion,decision,efecto
0,H1,Kruskal-Wallis,"< 0,001",Rechazar H0,0.127674
1,H2,Kruskal-Wallis,"< 0,001",Rechazar H0,0.231402
2,H2 completa vs privada,Mann-Whitney,"< 0,001",Rechazar H0,0.622642
3,H3,Spearman,"< 0,001",Rechazar H0,0.480350
4,H4,Spearman,"< 0,001",Rechazar H0,0.046257
5,H5 precio_mediano,Mann-Whitney,"< 0,001",Rechazar H0,0.074685
6,H5 ocupacion_media_dias,Mann-Whitney,"< 0,001",Rechazar H0,0.184722
7,H5 resenas_12m_media,Mann-Whitney,"< 0,001",Rechazar H0,0.212838
8,H5 calificacion_mediana,Mann-Whitney,"< 0,001",Rechazar H0,-0.199399


## 6. Sensibilidad exploratoria: un anuncio por anfitrión
**No es bootstrap ni intervalo de confianza.** Son 100 selecciones aleatorias (sin reemplazo) de un anuncio por anfitrión, con semilla fija.
Cada selección vuelve a exigir n ≥ 30 por grupo en H1 y H2.

In [7]:
CONTRASTES_GRUPOS = [("H1", "localidad"), ("H2", "tipo_alojamiento")]
CONTRASTES_CORRELACION = [("H3", "capacidad"), ("H4", "calificacion")]

rng = np.random.default_rng(SEED)
posiciones = list(d.reset_index(drop=True).groupby("id_host", sort=True).indices.values())
filas_sens = []
for iteracion in range(1, N_ITER_SENSIBILIDAD + 1):
    sub = d.iloc[[rng.choice(idx) for idx in posiciones]]
    for hip, campo in CONTRASTES_GRUPOS:
        sub_k, h, p, eta, k = kruskal(sub, campo)
        filas_sens.append([iteracion, hip, len(sub_k), k, eta, p])
    for hip, campo in CONTRASTES_CORRELACION:
        n, rho, p = spearman(sub, campo)
        filas_sens.append([iteracion, hip, n, np.nan, rho, p])

sens = pd.DataFrame(filas_sens, columns=["iteracion", "hipotesis", "n", "grupos", "efecto", "p"])
guardar_csv(sens, DIR_RESULTADOS, "sensibilidad_un_anuncio_por_host.csv")
resumen_sens = sens.groupby("hipotesis").agg(
    n_min=("n", "min"), n_max=("n", "max"), efecto_min=("efecto", "min"),
    efecto_mediano=("efecto", "median"), efecto_max=("efecto", "max"),
    selecciones_p_menor_005=("p", lambda s: int((s < .05).sum()))).reset_index()
guardar_csv(resumen_sens, DIR_RESULTADOS, "resumen_sensibilidad.csv")
resumen_sens

,hipotesis,n_min,n_max,efecto_min,efecto_mediano,efecto_max,selecciones_p_menor_005
0,H1,7701,7709,0.078667,0.082714,0.086433,100
1,H2,7767,7779,0.260376,0.265945,0.276022,100
2,H3,7795,7795,0.493953,0.501269,0.508814,100
3,H4,5917,5964,0.017915,0.036401,0.048397,96
